# Gated neural evaluation and final fit

This notebook verifies the immutable bundle, trains only missing 80-epoch ablations, runs paired bootstrap comparisons against `base` overall, in the mature period, and on the bookmaker intersection, then saves the evidence. Final fitting is disabled until an ablation is explicitly selected.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPOSITORY = 'https://github.com/RosarioDiBartolo/Sports-Predictions-Lab.git'
CODE_COMMIT = 'cd986fcc5cee77a58216a6be42b96a5bf4037385'
CHECKOUT_DIR = '/content/sports-predictions-lab'
BUNDLE_DIR = '/content/drive/MyDrive/ec301f01533c'
RUN_ROOT = '/content/drive/MyDrive/Sports Predictions Lab/runs'
EVIDENCE_ROOT = '/content/drive/MyDrive/Sports Predictions Lab/evidence'
FINAL_RUN_ROOT = '/content/drive/MyDrive/Sports Predictions Lab/final-runs'
EPOCHS = 80
BOOTSTRAP_SAMPLES = 5000
SEED = 42

In [ ]:
import pathlib
import shutil
import subprocess

checkout = pathlib.Path(CHECKOUT_DIR)
if checkout.exists():
    shutil.rmtree(checkout)
subprocess.run(['git', 'clone', REPOSITORY, CHECKOUT_DIR], check=True)
subprocess.run(['git', '-C', CHECKOUT_DIR, 'fetch', 'origin', 'codex/gated-final-fit'], check=True)
subprocess.run(['git', '-C', CHECKOUT_DIR, 'checkout', '--detach', CODE_COMMIT], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-e', CHECKOUT_DIR], check=True)

In [ ]:
import torch
from football_odds.modeling.training_bundle import verify_training_bundle

manifest = verify_training_bundle(pathlib.Path(BUNDLE_DIR))
print('Dataset:', manifest['dataset_version'])
print('Code:', CODE_COMMIT)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Select a Colab GPU runtime before training.')
print('GPU:', torch.cuda.get_device_name(0))

## Discover completed 80-epoch runs

In [ ]:
import json
from pathlib import Path

run_root = Path(RUN_ROOT)
model_name = 'dixon_coles_shared_encoder_pooling_gated'

def completed_runs(ablation, epochs=EPOCHS):
    found = []
    for run in run_root.iterdir():
        predictions = run / 'artifacts' / model_name / ablation / 'predictions.csv'
        preflight_path = run / 'preflight.json'
        state_path = run / 'run.json'
        if not (predictions.exists() and preflight_path.exists() and state_path.exists()):
            continue
        preflight = json.loads(preflight_path.read_text())
        state = json.loads(state_path.read_text())
        if preflight.get('passed') and preflight.get('config', {}).get('epochs') == epochs and state.get('status') == 'completed':
            found.append((run, predictions))
    return sorted(found, key=lambda item: item[0].stat().st_mtime, reverse=True)

for ablation in ('base', 'feature_store', 'bench', 'combined'):
    runs = completed_runs(ablation)
    print(ablation, runs[0][0].name if runs else 'MISSING')

## Train only missing feature-store and bench ablations

In [ ]:
import time

missing = [name for name in ('feature_store', 'bench') if not completed_runs(name)]
print('Missing:', missing)

if missing:
    command = [
        'odds-lab', '--project-dir', BUNDLE_DIR, 'model', 'compare',
        '--candidate', model_name,
        '--epochs', str(EPOCHS), '--max-iter', '20',
        '--device', 'cuda', '--run-root', RUN_ROOT,
    ]
    for ablation in missing:
        command.extend(['--ablation', ablation])

    existing = {path.name for path in run_root.iterdir() if path.is_dir()}
    print('Avvio:', ' '.join(command), flush=True)
    process = subprocess.Popen(command)
    active_run = None
    while process.poll() is None:
        new_runs = [path for path in run_root.iterdir() if path.is_dir() and path.name not in existing]
        if new_runs:
            active_run = max(new_runs, key=lambda path: path.stat().st_mtime)
            state_path = active_run / 'run.json'
            if state_path.exists():
                try:
                    state = json.loads(state_path.read_text())
                    print(f"status={state.get('status')} phase={state.get('phase')} ablation={state.get('ablation')} heartbeat={state.get('latest_heartbeat')}", flush=True)
                except json.JSONDecodeError:
                    print('run.json update in progress', flush=True)
            else:
                print('Preflight/input preparation:', active_run.name, flush=True)
        else:
            print('Waiting for run directory', flush=True)
        time.sleep(30)
    if process.wait() != 0:
        raise RuntimeError('Training process failed.')
    state = json.loads((active_run / 'run.json').read_text())
    if state.get('status') != 'completed':
        raise RuntimeError(f'Run did not complete: {state}')
    print('Completed:', active_run.name)
else:
    print('Both ablations are already complete; no training started.')

## Load all four prediction artifacts and validate pairing

In [ ]:
import pandas as pd

ablations = ('base', 'feature_store', 'bench', 'combined')
predictions = {}
selected_runs = {}

for ablation in ablations:
    runs = completed_runs(ablation)
    if not runs:
        raise RuntimeError(f'Missing completed 80-epoch run: {ablation}')
    run, path = runs[0]
    frame = pd.read_csv(path)
    frame['match_id'] = frame['match_id'].astype(str)
    frame['season'] = frame['season'].astype(str).str.zfill(4)
    if frame['match_id'].duplicated().any():
        raise RuntimeError(f'Duplicate predictions: {ablation}')
    predictions[ablation] = frame
    selected_runs[ablation] = run.name

base_ids = set(predictions['base']['match_id'])
for ablation, frame in predictions.items():
    if set(frame['match_id']) != base_ids:
        raise RuntimeError(f'Match intersection differs: {ablation}')
    outcomes = predictions['base'][['match_id', 'result']].merge(
        frame[['match_id', 'result']], on='match_id', suffixes=('_base', '_candidate')
    )
    if not (outcomes['result_base'] == outcomes['result_candidate']).all():
        raise RuntimeError(f'Outcome mismatch: {ablation}')

print('Paired matches:', len(base_ids))
print(json.dumps(selected_runs, indent=2))

## Paired bootstrap: overall, mature period, and bookmaker intersection

In [ ]:
from football_odds.modeling.evaluation import paired_log_loss_bootstrap

snapshot_path = Path(BUNDLE_DIR) / 'data/raw/beat_the_bookie/reconciled_cutoff_snapshot.csv'
snapshot = pd.read_csv(snapshot_path, usecols=['match_id'])
bookmaker_ids = set(snapshot['match_id'].astype(str))

base = predictions['base']
base_mature = base[base['season'] >= '1920']
base_bookmaker = base[base['match_id'].isin(bookmaker_ids)]
print('Bookmaker intersection:', len(base_bookmaker))

evidence = {
    'reference': 'base',
    'epochs': EPOCHS,
    'bootstrap_samples': BOOTSTRAP_SAMPLES,
    'seed': SEED,
    'dataset_version': manifest['dataset_version'],
    'code_version': CODE_COMMIT,
    'runs': selected_runs,
    'comparisons': {},
}

for ablation in ('feature_store', 'bench', 'combined'):
    candidate = predictions[ablation]
    result = {
        'overall': paired_log_loss_bootstrap(candidate, base, samples=BOOTSTRAP_SAMPLES, seed=SEED),
        'mature_2019_20_to_2024_25': paired_log_loss_bootstrap(
            candidate[candidate['season'] >= '1920'], base_mature,
            samples=BOOTSTRAP_SAMPLES, seed=SEED,
        ),
        'bookmaker_intersection': paired_log_loss_bootstrap(
            candidate[candidate['match_id'].isin(bookmaker_ids)], base_bookmaker,
            samples=BOOTSTRAP_SAMPLES, seed=SEED,
        ),
    }
    evidence['comparisons'][ablation] = result
    print('\n', ablation)
    print(json.dumps(result, indent=2))

## Save immutable comparison evidence

In [ ]:
from datetime import datetime, timezone

evidence_root = Path(EVIDENCE_ROOT)
evidence_root.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
evidence_path = evidence_root / f'gated_ablation_bootstrap_{stamp}.json'
evidence_path.write_text(json.dumps(evidence, indent=2), encoding='utf-8')
print('Saved:', evidence_path)

## Final fit — deliberately disabled

Set `FINAL_ABLATION` only after reviewing all evidence. Setting `RUN_FINAL_FIT=True` is an explicit decision to create a non-operational candidate artifact; it does not promote the model.

In [ ]:
RUN_FINAL_FIT = False
FINAL_ABLATION = None  # 'base', 'feature_store', 'bench', or 'combined'

if RUN_FINAL_FIT:
    if FINAL_ABLATION not in ablations:
        raise ValueError('Select a valid FINAL_ABLATION after evidence review.')
    command = [
        'odds-lab', '--project-dir', BUNDLE_DIR,
        'model', 'fit-final-gated',
        '--ablation', FINAL_ABLATION,
        '--epochs', str(EPOCHS),
        '--device', 'cuda',
        '--code-version', CODE_COMMIT,
        '--run-root', FINAL_RUN_ROOT,
    ]
    print('Avvio fit finale:', ' '.join(command), flush=True)
    subprocess.run(command, check=True)
else:
    print('Final fit disabled. Review the saved bootstrap evidence first.')